# 0.2 Reduce spectra, one at a time

Reduction order: **clip → flag bad pixels → adjust uncertainties → smooth**. This notebook does the **clipping** step only so far.

For each spectrum:
1. **Propose:** the finders shade candidate regions on the raw spectrum. Red = noise spikes, orange = low S/N, blue = telluric bands, dark grey = trimmed ends. The S/N panel underneath shows why each was chosen.
2. **Edit:** start from the proposed lists and widen, shrink, delete or add regions by hand. Each region is just a `(wl_lo, wl_hi)` tuple in Å.
3. **Check:** in the clipped plot, grey is raw, black is kept, and red is interpolated. Interpolated pixels get `weight = 0`.
4. **Save:** writes `Inputs/Spectroscopy/1.5_spec_clipped/<SN>/<file>` with columns `wls flux flux_err weight`. It also records your regions and edges in `<SN>_clip_regions.json` and updates `1.5_spec_lists_clipped/<SN>.list`.

Ends outside `edges` are removed outright. Interior regions are interpolated rather than removed, (a straight line between the median flux within 10 Å on each side), so the spectrum has no gaps; the zero weights are what tell later steps to ignore them.

All functions live in `helper_scripts/spec_reduce.py`. Every plot function takes an `ax`, so you can change a single plot (`ax.set_xlim(...)`, `ax.set_ylim(...)`) without touching anything else.

In [ ]:
import pipeline_config as pconf
pconf.ensure_helper_paths()

import numpy as np
import matplotlib.pyplot as plt
import spec_reduce as sr

%matplotlib inline

SN = "AT2017gfo"
files = sr.list_spectra(SN)          # [(mjd, filename), ...] sorted by MJD
for i, (mjd, f) in enumerate(files):
    print("%2d  %.3f  %s" % (i, mjd, f))

## Default settings

Copied into `params` for each spectrum, so you can override any of them for one spectrum only.

- `spike_*`: a pixel is flagged when it sits more than `spike_nsigma` × the local scatter away from the running median over `spike_window_A`. Each flagged pixel is grown by `spike_grow_A` on both sides.
- `snr_*`: S/N is measured **per `snr_bin_A` Å**, so XSHOOTER (0.2–0.6 Å pixels) and LDSS3 (2 Å pixels) can share one threshold. It uses the input errors when the file has them (`use_input_err`). Otherwise it estimates the noise from the scatter about a running median.
- `low_snr_min_width_A`: ignore low-S/N dips narrower than this.
- `edge_hold_A`: each end is cut where S/N first and last stays above `snr_min` for at least this many Å.

In [ ]:
DEFAULTS = dict(
    spike_window_A=50.0,
    spike_nsigma=5.0,
    spike_grow_A=2.0,
    snr_min=5.0,
    snr_bin_A=10.0,
    snr_window_A=50.0,
    use_input_err=True,
    low_snr_min_width_A=20.0,
    edge_hold_A=100.0,
    telluric_bands=sr.TELLURIC_BANDS,
)
sr.TELLURIC_BANDS

---
## 0: `57983.01825_LDSS3_Magellan.dat`
MJD 57983.018

In [ ]:
i = 0
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 1: `57983.02788_LDSS3_Magellan.dat`
MJD 57983.028

In [ ]:
i = 1
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 2: `57983.0362_LDSS3_Magellan.dat`
MJD 57983.036

In [ ]:
i = 2
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 3: `57983.05998_LDSS3_Magellan.dat`
MJD 57983.060

In [ ]:
i = 3
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 4: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969001_Phase+1.43d.dat`
MJD 57983.969

In [ ]:
i = 4
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 5: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969002_Phase+1.43d.dat`
MJD 57983.969

In [ ]:
i = 5
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 6: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969003_Phase+1.43d.dat`
MJD 57983.969

In [ ]:
i = 6
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 7: `57983.9914_LDSS3_Magellan.dat`
MJD 57983.991

In [ ]:
i = 7
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 8: `57984.0484_LDSS3_Magellan.dat`
MJD 57984.048

In [ ]:
i = 8
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 9: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57984.969001_Phase+2.42d.dat`
MJD 57984.969

In [ ]:
i = 9
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 10: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57984.969002_Phase+2.42d.dat`
MJD 57984.969

In [ ]:
i = 10
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 11: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57984.969003_Phase+2.42d.dat`
MJD 57984.969

In [ ]:
i = 11
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 12: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57985.974001_Phase+3.41d.dat`
MJD 57985.974

In [ ]:
i = 12
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 13: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57985.974002_Phase+3.41d.dat`
MJD 57985.974

In [ ]:
i = 13
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 14: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57985.974003_Phase+3.41d.dat`
MJD 57985.974

In [ ]:
i = 14
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 15: `57985.99019_LDSS3_Magellan.dat`
MJD 57985.990

In [ ]:
i = 15
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 16: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57986.974001_Phase+4.40d.dat`
MJD 57986.974

In [ ]:
i = 16
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 17: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57986.974002_Phase+4.40d.dat`
MJD 57986.974

In [ ]:
i = 17
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 18: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57986.974003_Phase+4.40d.dat`
MJD 57986.974

In [ ]:
i = 18
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 19: `57987.03511_LDSS3_Magellan.dat`
MJD 57987.035

In [ ]:
i = 19
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 20: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57987.980001_Phase+5.40d.dat`
MJD 57987.980

In [ ]:
i = 20
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 21: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57987.980002_Phase+5.40d.dat`
MJD 57987.980

In [ ]:
i = 21
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 22: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57987.980003_Phase+5.40d.dat`
MJD 57987.980

In [ ]:
i = 22
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 23: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57988.990001_Phase+6.40d.dat`
MJD 57988.990

In [ ]:
i = 23
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 24: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57988.990002_Phase+6.40d.dat`
MJD 57988.990

In [ ]:
i = 24
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 25: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57988.990003_Phase+6.40d.dat`
MJD 57988.990

In [ ]:
i = 25
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 26: `57989.98185_LDSS3_Magellan.dat`
MJD 57989.982

In [ ]:
i = 26
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 27: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57990.000001_Phase+7.40d.dat`
MJD 57990.000

In [ ]:
i = 27
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 28: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57990.000002_Phase+7.40d.dat`
MJD 57990.000

In [ ]:
i = 28
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 29: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57990.000003_Phase+7.40d.dat`
MJD 57990.000

In [ ]:
i = 29
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 30: `57990.98563_LDSS3_Magellan.dat`
MJD 57990.986

In [ ]:
i = 30
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 31: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57991.000001_Phase+8.40d.dat`
MJD 57991.000

In [ ]:
i = 31
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 32: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57991.000003_Phase+8.40d.dat`
MJD 57991.000

In [ ]:
i = 32
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 33: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57992.000001_Phase+9.40d.dat`
MJD 57992.000

In [ ]:
i = 33
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 34: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57992.000002_Phase+9.40d.dat`
MJD 57992.000

In [ ]:
i = 34
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 35: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57992.000003_Phase+9.40d.dat`
MJD 57992.000

In [ ]:
i = 35
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 36: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57993.000001_Phase+10.40d.dat`
MJD 57993.000

In [ ]:
i = 36
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 37: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57993.000002_Phase+10.40d.dat`
MJD 57993.000

In [ ]:
i = 37
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## 38: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57993.000003_Phase+10.40d.dat`
MJD 57993.000

In [ ]:
i = 38
mjd, fname = files[i]
spec = sr.load_raw(SN, fname)
params = dict(DEFAULTS)              # e.g. params["spike_nsigma"] = 7
proposed, edges = sr.propose_clips(spec, params)
print({k: len(v) for k, v in proposed.items()}, "edges:", edges)

fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
sr.plot_raw(spec, axs[0])
sr.plot_proposed(spec, proposed, edges, axs[1])
sr.plot_snr(spec, params, axs[2])
plt.show()

In [ ]:
# Edit what gets clipped. Each region is a (wl_lo, wl_hi) tuple in Angstrom.
regions = proposed["spike"] + proposed["low_snr"] + proposed["telluric"]
# regions = sr.remove_between(regions, 5000, 6000)   # drop proposals in a range
manual = []                                           # e.g. [(7590, 7700)]
# edges = (3800, 10000)                               # override the trimmed ends
# saved = sr.load_choices(SN, fname); regions, edges, manual = saved["regions"], saved["edges"], []

clipped = sr.apply_clip(spec, regions + manual, edges)

fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
sr.plot_proposed(spec, {"clip": regions, "manual": manual}, edges, axs[0])
sr.plot_clipped(spec, clipped, axs[1])
# axs[1].set_xlim(7000, 8000)                         # zoom in
plt.show()

In [ ]:
# Run this only when you're happy with the result above.
# sr.save_clipped(SN, mjd, clipped, regions + manual, edges, note="")

---
## Overview of what's been saved

In [ ]:
choices = sr.load_choices(SN)
for name, c in sorted(choices.items(), key=lambda kv: kv[1]["mjd"]):
    print("%.3f  %-70s  edges %.0f-%.0f  %3d regions  %s"
          % (c["mjd"], name, c["edges"][0], c["edges"][1], len(c["regions"]), c["note"]))
print(len(choices), "of", len(files), "spectra saved")